# Flight Risk Prediction Model

## Objective
Build a predictive model to identify employees at high risk of leaving before they resign.

## Key Questions
1. Which employees are most likely to leave in the next 6-12 months?
2. What factors are most predictive of attrition?
3. How accurate is the model?
4. Who should HR prioritize for retention efforts?

## Approach
- **Model Type**: Classification (Random Forest)
- **Target**: Departed (Yes/No) based on historical data
- **Features**: Compensation, engagement, performance, tenure, promotion history
- **Output**: Flight risk score (0-100) for each active employee

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Loading data...")
employees = pd.read_csv('../data/employees.csv')
print(f"Loaded {len(employees)} employee records")

## 1. Prepare Data for Modeling

In [ ]:
# Create target variable: Did they leave? (1 = Yes, 0 = No)
employees['departed'] = (employees['status'] == 'Terminated').astype(int)

# Select features for the model
feature_cols = [
    'tenure_years',
    'market_percentile',
    'months_since_promotion',
    'months_since_raise',
    'performance_rating',
    'engagement_score',
    'manager_tenure_years',
    'high_performer',
    'critical_role'
]

# Create additional features
# Department as dummy variables
dept_dummies = pd.get_dummies(employees['department'], prefix='dept')

# Job level as dummy variables
level_dummies = pd.get_dummies(employees['job_level'], prefix='level')

# Location as dummy variables (Remote vs. Office)
employees['is_remote'] = (employees['location'] == 'Remote').astype(int)

# Combine all features
X = pd.concat([
    employees[feature_cols],
    dept_dummies,
    level_dummies,
    employees[['is_remote']]
], axis=1)

y = employees['departed']

print(f"\nFeatures: {X.shape[1]} total")
print(f"Target distribution:")
print(f"  Stayed: {(y==0).sum()} ({(y==0).sum()/len(y)*100:.1f}%)")
print(f"  Departed: {(y==1).sum()} ({(y==1).sum()/len(y)*100:.1f}%)")

## 2. Train-Test Split and Model Training

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training set: {len(X_train)} employees")
print(f"Test set: {len(X_test)} employees")

# Train Random Forest model
print("\nTraining Random Forest model...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    class_weight='balanced'  # Handle class imbalance
)

rf_model.fit(X_train, y_train)
print("Model training complete!")

## 3. Model Evaluation

In [ ]:
# Predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Calculate metrics
auc_score = roc_auc_score(y_test, y_pred_proba)

print("MODEL PERFORMANCE")
print("="*60)
print(f"AUC-ROC Score: {auc_score:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Departed']))
print("="*60)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print("              Predicted Stay  Predicted Depart")
print(f"Actual Stay        {cm[0,0]:4d}           {cm[0,1]:4d}")
print(f"Actual Depart      {cm[1,0]:4d}           {cm[1,1]:4d}")

In [ ]:
# Visualize model performance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
ax1.plot(fpr, tpr, linewidth=2, label=f'AUC = {auc_score:.3f}')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Guess')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['Predicted Stay', 'Predicted Depart'],
            yticklabels=['Actual Stay', 'Actual Depart'])
ax2.set_title('Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Feature Importance Analysis

In [ ]:
# Get feature importances
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Top 15 features
top_features = feature_importance.head(15)

print("TOP 15 PREDICTIVE FEATURES")
print("="*60)
for idx, row in top_features.iterrows():
    print(f"{row['feature']:35s} {row['importance']:.4f}")
print("="*60)

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(range(len(top_features)), top_features['importance'], color='#2ecc71', alpha=0.7)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Top 15 Predictive Features for Attrition', fontsize=14, fontweight='bold')

# Add value labels
for i, val in enumerate(top_features['importance']):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Generate Flight Risk Scores for Active Employees

In [ ]:
# Get active employees only
active_employees = employees[employees['status'] == 'Active'].copy()
print(f"Scoring {len(active_employees)} active employees...")

# Prepare features for active employees (same as training)
dept_dummies_active = pd.get_dummies(active_employees['department'], prefix='dept')
level_dummies_active = pd.get_dummies(active_employees['job_level'], prefix='level')
active_employees['is_remote'] = (active_employees['location'] == 'Remote').astype(int)

X_active = pd.concat([
    active_employees[feature_cols].reset_index(drop=True),
    dept_dummies_active.reset_index(drop=True),
    level_dummies_active.reset_index(drop=True),
    active_employees[['is_remote']].reset_index(drop=True)
], axis=1)

# Ensure columns match training data
for col in X.columns:
    if col not in X_active.columns:
        X_active[col] = 0
X_active = X_active[X.columns]

# Generate flight risk scores (probability of departure × 100)
flight_risk_proba = rf_model.predict_proba(X_active)[:, 1]
active_employees['flight_risk_score'] = (flight_risk_proba * 100).round(1)

# Assign risk tiers
def risk_tier(score):
    if score >= 60:
        return 'High Risk'
    elif score >= 40:
        return 'Medium Risk'
    else:
        return 'Low Risk'

active_employees['risk_tier'] = active_employees['flight_risk_score'].apply(risk_tier)

print("\nFLIGHT RISK DISTRIBUTION")
print("="*60)
risk_counts = active_employees['risk_tier'].value_counts()
for tier in ['High Risk', 'Medium Risk', 'Low Risk']:
    count = risk_counts.get(tier, 0)
    pct = (count / len(active_employees)) * 100
    print(f"{tier:15s}: {count:3d} employees ({pct:.1f}%)")
print("="*60)

In [ ]:
# Visualize flight risk distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Histogram of risk scores
ax1.hist(active_employees['flight_risk_score'], bins=20, color='#e74c3c', alpha=0.7, edgecolor='black')
ax1.axvline(40, color='orange', linestyle='--', linewidth=2, label='Medium Risk Threshold')
ax1.axvline(60, color='red', linestyle='--', linewidth=2, label='High Risk Threshold')
ax1.set_xlabel('Flight Risk Score', fontsize=12)
ax1.set_ylabel('Number of Employees', fontsize=12)
ax1.set_title('Distribution of Flight Risk Scores', fontsize=14, fontweight='bold')
ax1.legend()

# Pie chart: Risk tiers
tier_order = ['High Risk', 'Medium Risk', 'Low Risk']
tier_counts = [risk_counts.get(tier, 0) for tier in tier_order]
colors_pie = ['#e74c3c', '#f39c12', '#2ecc71']

ax2.pie(tier_counts, labels=tier_order, autopct='%1.1f%%', colors=colors_pie,
        startangle=90, explode=(0.1, 0, 0))
ax2.set_title('Active Employees by Risk Tier', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. High-Risk Employee List

In [ ]:
# Get high-risk employees
high_risk = active_employees[active_employees['risk_tier'] == 'High Risk'].copy()
high_risk = high_risk.sort_values('flight_risk_score', ascending=False)

# Select relevant columns for the list
high_risk_display = high_risk[[
    'employee_id', 'department', 'job_level', 'tenure_years',
    'performance_rating', 'engagement_score', 'market_percentile',
    'months_since_promotion', 'high_performer', 'critical_role',
    'flight_risk_score'
]].head(20)

print("\nTOP 20 HIGH-RISK EMPLOYEES")
print("="*100)
print(high_risk_display.to_string(index=False))
print("="*100)

# Identify critical high-risk employees
critical_high_risk = high_risk[(high_risk['high_performer'] == 1) | (high_risk['critical_role'] == 1)]
print(f"\n⚠️  CRITICAL ALERT: {len(critical_high_risk)} high-risk employees are either")
print("   high performers or in critical roles!")
print("   → These should be prioritized for immediate retention conversations")

## 7. Risk Profile Analysis

In [ ]:
# Compare characteristics across risk tiers
risk_profile = active_employees.groupby('risk_tier').agg({
    'employee_id': 'count',
    'engagement_score': 'mean',
    'market_percentile': 'mean',
    'months_since_promotion': 'mean',
    'months_since_raise': 'mean',
    'performance_rating': 'mean',
    'tenure_years': 'mean'
}).round(2)

risk_profile.columns = ['Count', 'Avg Engagement', 'Avg Market %ile', 
                        'Avg Months Since Promo', 'Avg Months Since Raise',
                        'Avg Performance', 'Avg Tenure']

risk_profile = risk_profile.reindex(['High Risk', 'Medium Risk', 'Low Risk'])

print("\nRISK PROFILE COMPARISON")
print("="*90)
print(risk_profile)
print("="*90)

print("\nKEY PATTERNS:")
high_risk_profile = risk_profile.loc['High Risk']
low_risk_profile = risk_profile.loc['Low Risk']

if high_risk_profile['Avg Engagement'] < low_risk_profile['Avg Engagement'] - 10:
    print("✓ High-risk employees have significantly LOWER engagement")
if high_risk_profile['Avg Market %ile'] < low_risk_profile['Avg Market %ile'] - 5:
    print("✓ High-risk employees are paid BELOW market compared to low-risk")
if high_risk_profile['Avg Months Since Promo'] > low_risk_profile['Avg Months Since Promo'] + 3:
    print("✓ High-risk employees went LONGER without promotion")

In [ ]:
# Visualize risk profile comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

metrics = ['Avg Engagement', 'Avg Market %ile', 'Avg Months Since Promo', 
           'Avg Months Since Raise', 'Avg Performance', 'Avg Tenure']
colors_bars = ['#e74c3c', '#f39c12', '#2ecc71']

for i, metric in enumerate(metrics):
    values = [risk_profile.loc[tier, metric] for tier in ['High Risk', 'Medium Risk', 'Low Risk']]
    bars = axes[i].bar(['High', 'Medium', 'Low'], values, color=colors_bars, alpha=0.7)
    axes[i].set_title(metric, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Score')
    
    # Add value labels
    for j, val in enumerate(values):
        axes[i].text(j, val + max(values)*0.02, f'{val:.1f}', 
                    ha='center', fontweight='bold', fontsize=10)

plt.suptitle('Risk Profile Comparison Across Tiers', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 8. Department-Level Flight Risk

In [ ]:
# Calculate average flight risk by department
dept_risk = active_employees.groupby('department').agg({
    'employee_id': 'count',
    'flight_risk_score': 'mean',
    'risk_tier': lambda x: (x == 'High Risk').sum()
}).round(1)

dept_risk.columns = ['Employee Count', 'Avg Flight Risk', 'High Risk Count']
dept_risk['% High Risk'] = (dept_risk['High Risk Count'] / dept_risk['Employee Count'] * 100).round(1)
dept_risk = dept_risk.sort_values('Avg Flight Risk', ascending=False)

print("FLIGHT RISK BY DEPARTMENT")
print("="*70)
print(dept_risk)
print("="*70)

In [ ]:
# Visualize department flight risk
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(range(len(dept_risk)), dept_risk['Avg Flight Risk'], 
               color=['#e74c3c' if x > 45 else '#3498db' for x in dept_risk['Avg Flight Risk']],
               alpha=0.7)
ax.set_yticks(range(len(dept_risk)))
ax.set_yticklabels(dept_risk.index)
ax.set_xlabel('Average Flight Risk Score', fontsize=12)
ax.set_title('Average Flight Risk by Department', fontsize=14, fontweight='bold')

# Add value labels
for i, (idx, row) in enumerate(dept_risk.iterrows()):
    ax.text(row['Avg Flight Risk'] + 1, i, 
            f"{row['Avg Flight Risk']:.1f} ({int(row['High Risk Count'])} high risk)", 
            va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Recommended Actions

In [ ]:
print("\n" + "="*80)
print("RECOMMENDED ACTIONS")
print("="*80)

print("\n1. IMMEDIATE (This Week)")
print("   → Share high-risk list with HR Business Partners")
print(f"   → Schedule stay conversations with {len(critical_high_risk)} critical high-risk employees")
print("   → Review compensation for high-risk high performers below market")

print("\n2. SHORT-TERM (This Month)")
print("   → Conduct engagement pulse survey for high-risk population")
print("   → Review promotion pipeline (address long times since promotion)")
print("   → Manager coaching for teams with high average risk scores")

print("\n3. ONGOING")
print("   → Refresh flight risk scores monthly")
print("   → Track retention rate of high-risk employees (measure intervention impact)")
print("   → Retrain model quarterly with new data")

print("\n4. FOCUS AREAS (Based on Feature Importance)")
top_3_features = top_features.head(3)['feature'].tolist()
for i, feat in enumerate(top_3_features, 1):
    print(f"   {i}. Address '{feat}' as a top predictor of attrition")

print("\n" + "="*80)
print("Next Analysis: See '03_retention_drivers.ipynb' for deep dive")
print("into root causes and exit interview analysis")
print("="*80)

## 10. Export High-Risk List

In [ ]:
# Save high-risk employee list
high_risk_export = high_risk[[
    'employee_id', 'department', 'job_level', 'location',
    'tenure_years', 'performance_rating', 'engagement_score',
    'market_percentile', 'months_since_promotion', 'months_since_raise',
    'high_performer', 'critical_role', 'flight_risk_score', 'risk_tier'
]].sort_values('flight_risk_score', ascending=False)

output_file = '../data/high_risk_employees.csv'
high_risk_export.to_csv(output_file, index=False)
print(f"\n✅ High-risk employee list saved to: {output_file}")
print(f"   Total high-risk employees: {len(high_risk_export)}")